<a href="https://colab.research.google.com/github/rickiepark/llm-from-scratch/blob/main/ch07/01_main-chapter-code/ch07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
세바스찬 라시카(Sebastian Raschka)가 쓴 <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a>의 번역서 <br><<b><a href="<a href="http://tensorflow.blog/llm-from-scratch">밑바닥부터 만들면서 배우는 LLM</a></b>>의 예제 코드입니다.<br>
<br>코드 저장소: <a href="https://github.com/rickiepark/llm-from-scratch">https://github.com/rickiepark/llm-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://tensorflow.blog/llm-from-scratch"><img src="https://tensorflowkorea.wordpress.com/wp-content/uploads/2025/09/ebb091ebb094eb8ba5llm_ebb3b8ecb185_ec959eeba9b4.jpg" width="100px"></a>
</td>
</tr>
</table>

# 7장: 지시를 따르도록 미세 튜닝하기


In [ ]:
from importlib.metadata import version

pkgs = [
    "numpy",       # 파이토치와 텐서플로 의존성
    "matplotlib",  # 그래프 라이브러리
    "tiktoken",    # 토크나이저
    "torch",       # 딥러닝 라이브러리
    "tqdm",        # 진행 표시줄
]
for p in pkgs:
    print(f"{p} 버전: {version(p)}")

<img src="images/llm_from_scratch/ch07_compressed/01.webp" width=800px>


## 7.1 지시 미세 튜닝 소개


- 5장에서, 우리는 LLM 사전 훈련이 한 번에 한 단어씩 생성하는 것을 배우는 것임을 보았습니다.
- 따라서 사전 훈련된 LLM은 텍스트 완성에는 능숙하지만 지시 사항을 따르는 데는 능숙하지 않습니다.
- 이 장에서는 LLM이 지시 사항을 더 잘 따르도록 가르칩니다.


<img src="images/llm_from_scratch/ch07_compressed/02.webp" width=600px>


- 이 장에서 다루는 주제는 아래 그림에 요약되어 있습니다.

<img src="images/llm_from_scratch/ch07_compressed/03.webp" width=700px>


## 7.2 지도 학습 미세 튜닝을 위한 데이터셋 준비하기


- 이 장을 위해 준비한 지시 데이터셋을 사용합니다.


In [ ]:
import json
import os
import requests


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data

file_path = "datas/instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("샘플 개수:", len(data))

- 위 JSON 파일에서 로드한 `data` 리스트의 각 항목은 다음 형식의 딕셔너리입니다.

In [ ]:
print("샘플 예시:\n", data[50])

- `'input'` 필드는 비어 있을 수 있습니다.


In [ ]:
print("다른 샘플:\n", data[999])

- 지시 미세 튜닝은 입력-출력 쌍이 명시적으로 제공되는 데이터셋에서 모델을 학습시키기 때문에 종종 "지도 학습 지시 미세 튜닝"이라고 합니다.
- LLM에 입력으로 항목을 포맷하는 다양한 방법이 있습니다. 아래 그림은 각각 알파카(https://crfm.stanford.edu/2023/03/13/alpaca.html) 및 Phi-3(https://arxiv.org/abs/2404.14219) LLM 훈련에 사용된 두 가지 예시 형식을 보여줍니다.


<img src="images/llm_from_scratch/ch07_compressed/04.webp" width=800px>


- 이 장에서는 지시 미세 튜닝을 위한 초기 프롬프트 템플릿인 알파카 스타일 프롬프트 형식을 사용합니다.
- 아래에서는 LLM에 입력으로 전달할 입력을 포맷팅합니다.


In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

- 입력 필드가 있는 경우의 출력은 아래와 같습니다.


In [ ]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"

print(model_input + desired_response)

- 입력 필드가 없는 경우는 다음과 같습니다.


In [ ]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"

print(model_input + desired_response)

- 마지막으로 다음 절에서 파이토치 데이터 로더를 준비하기 전에 데이터 세트를 훈련, 검증 및 테스트 세트로 나눕니다.


In [ ]:
train_portion = int(len(data) * 0.85)  # 훈련을 위한 85%
test_portion = int(len(data) * 0.1)    # 테스트를 위한 10%
val_portion = len(data) - train_portion - test_portion  # 나머지 5%는 검증용

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [ ]:
print("훈련 세트 길이:", len(train_data))
print("검증 세트 길이:", len(val_data))
print("테스트 세트 길이:", len(test_data))

## 7.3 훈련 배치 만들기


<img src="images/llm_from_scratch/ch07_compressed/05.webp" width=700px>


- 다음 그림에 요약된 것처럼 여러 단계로 이 데이터셋 배치를 처리합니다.

<img src="images/llm_from_scratch/ch07_compressed/06.webp" width=700px>


- 먼저, 6장의 `SpamDataset`과 유사하게 데이터셋의 모든 입력을 토큰화하는 `InstructionDataset` 클래스를 구현합니다.

<img src="images/llm_from_scratch/ch07_compressed/07.webp" width=800px>


In [ ]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # 텍스트 토큰화
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

- 6장과 유사하게, 훈련 속도를 높이기 위해 배치 단위로 여러 훈련 샘플을 묶습니다. 이를 위해서는 모든 입력을 비슷한 길이로 패딩해야 합니다.
- 또한 이전 장과 마찬가지로 `<|endoftext|>` 토큰을 패딩 토큰으로 사용합니다.


In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

- 6장에서는 데이터셋의 모든 샘플을 같은 길이로 패딩했습니다.
  - 여기서는 더 정교한 접근 방식을 취하고 데이터 로더에 전달할 수 있는 사용자 지정 콜레이트 함수를 개발합니다.
  - 이 사용자 지정 콜레이트 함수는 각 배치의 훈련 샘플을 동일한 길이로 패딩합니다(단, 배치마다 길이가 다를 수 있음).


<img src="images/llm_from_scratch/ch07_compressed/08.webp" width=800px>


In [ ]:
def custom_collate_draft_1(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # 배치에서 가장 긴 시퀀스 찾기
    # 그리고 최대 길이를 +1씩 증가시켜 아래에서 패딩 토큰을 하나 추가합니다.
    batch_max_length = max(len(item)+1 for item in batch)

    # 입력 패딩 및 준비
    inputs_lst = []

    for item in batch:
        new_item = item.copy()
        # <|endoftext|> 토큰 추가
        new_item += [pad_token_id]
        # batch_max_length까지 시퀀스 패딩
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        # padded[:-1]를 통해 batch_max_length의 +1 설정을 통해 추가된
        # 추가 패딩 토큰을 제거합니다.
        # (추가 패딩 토큰은 이후 코드에서 관련이 있습니다)
        inputs = torch.tensor(padded[:-1])
        inputs_lst.append(inputs)

    # 입력 리스트를 텐서로 변환하고 타깃 장치로 전송
    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

In [ ]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

print(custom_collate_draft_1(batch))

<img src="images/llm_from_scratch/ch07_compressed/09.webp" width=700px>


- 위에서는 LLM에 대한 입력만 반환했습니다. 그러나 LLM 훈련을 위해서는 타깃 값도 필요합니다.
- LLM 사전 훈련과 유사하게, 타깃은 입력을 오른쪽으로 1 위치씩 이동한 것이므로 LLM은 다음 토큰을 예측하는 방법을 학습할 수 있습니다.


<img src="images/llm_from_scratch/ch07_compressed/10.webp" width=600px>


In [ ]:
def custom_collate_draft_2(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # 배치에서 가장 긴 시퀀스 찾기
    batch_max_length = max(len(item)+1 for item in batch)

    # 입력 및 타깃 준비
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # <|endoftext|> 토큰 추가
        new_item += [pad_token_id]
        # 시퀀스를 max_length까지 패딩
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # 입력을 위해 마지막 토큰 자르기
        targets = torch.tensor(padded[1:])  # 타깃을 위해 오른쪽으로 +1 이동
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # 입력 리스트를 텐서로 변환하고 타깃 장치로 전송
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [ ]:
inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)

- 다음으로, 모든 패딩 토큰 ID를 `ignore_index`에 지정한 값으로 바꿉니다. 이 `ignore_index`의 목적은 손실 함수에서 패딩 값을 무시할 수 있도록 하는 것입니다(자세한 내용은 나중에 설명).

<img src="images/llm_from_scratch/ch07_compressed/11.webp" width=700px>

- 구체적으로, 아래 그림과 같이 `50256`에 해당하는 토큰 ID를 `-100`으로 바꿉니다.


<img src="images/llm_from_scratch/ch07_compressed/12.webp" width=800px>


- (또한, 샘플의 길이를 제한하고 싶은 경우를 위해 `allowed_max_length`를 사용했습니다. GPT-2 모델에서 지원하는 1024 토큰 문맥 크기보다 긴 데이터셋으로 작업하려는 경우 유용합니다.)


In [ ]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # 배치에서 가장 긴 시퀀스 찾기
    batch_max_length = max(len(item)+1 for item in batch)

    # 입력과 타깃 패딩 및 준비
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # <|endoftext|> 토큰 추가
        new_item += [pad_token_id]
        # 시퀀스를 max_length까지 패딩
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # 입력을 위해 마지막 토큰 자르기
        targets = torch.tensor(padded[1:])  # 목표를 위해 오른쪽으로 +1 이동

        # 새로 추가: 목표에서 첫 번째 패딩 토큰을 제외한 모든 토큰을 ignore_index로 바꾸기
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # 새로 추가: 최대 시퀀스 길이로 자르기 (선택 사항)
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # 입력 및 타깃 리스트를 텐서로 변환하고 타깃 장치로 전송
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [ ]:
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

- -100으로 대체하면 어떻게 되는지 살펴보겠습니다.
- 설명을 위해 6장과 유사하게 클래스 레이블이 0과 1인 2개의 작은 분류 작업이 있다고 가정해 보겠습니다.
- 다음과 같은 로짓 값(모델 마지막 층의 출력)이 있는 경우 다음과 같은 손실을 계산합니다.


In [ ]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  # 첫 번째 훈련 샘플
     [-0.5, 1.5]]  # 두 번째 훈련 샘플
)
targets_1 = torch.tensor([0, 1])

loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

가장 중요한 점은 모델이 첫 번째 문제를 틀렸다는 것입니다.
- logits_1 (모델의 예측 점수): 3행 2열의 텐서입니다.
  - 첫 번째 샘플: [-1.0, 1.0] → 모델은 1번 클래스일 확률이 높다고 예측함 (1.0이 더 크니까).
  - 두 번째 샘플: [-0.5, 1.5] → 모델은 1번 클래스라고 예측함.

- targets_1 (실제 정답): [0, 1]
  - 첫 번째 정답: 0번 클래스
  - 두 번째 정답: 1번 클래스

- 이제, 예상대로 하나의 훈련 샘플을 더 추가하면 손실에 영향을 미칩니다.


In [ ]:
logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # 새로운 세 번째 훈련 샘플
)
targets_2 = torch.tensor([0, 1, 1])

loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

- 한 샘플의 클래스 레이블을 -100으로 바꾸면 어떻게 되는지 봅시다.


In [ ]:
targets_3 = torch.tensor([0, 1, -100])

loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

- 여기서 볼 수 있듯이, 3개의 훈련 샘플에 대한 손실은 2개의 훈련 샘플에서 계산한 손실과 동일합니다. 이는 크로스 엔트로피 손실 함수가 -100 레이블이 있는 훈련 샘플을 무시했음을 의미합니다.
- 기본적으로 파이토치는 레이블 -100에 해당하는 샘플을 무시하도록 `cross_entropy(..., ignore_index=-100)` 설정을 사용합니다.
- -100인 `ignore_index`를 사용하여 훈련 샘플을 동일한 길이로 채우는 데 사용했던 배치의 추가적인 텍스트 끝(패딩) 토큰을 무시할 수 있습니다.
- 그러나 텍스트 끝(패딩) 토큰(50256)의 첫 번째는 LLM에 응답이 완료되었음을 알리는 데 도움이 될 수 있으므로 무시하고 싶지 않습니다.


- 실제로 아래 그림과 같이 지시 사항에 해당하는 대상 토큰 ID를 마스킹하는 것이 일반적입니다(이 장을 끝낸 독자를 위해 연습문제 남겨 놓겠습니다).


<img src="images/llm_from_scratch/ch07_compressed/13.webp" width=800px>


## 7.4 지시 데이터셋을 위한 데이터 로더 만들기


- 이 절에서는 `InstructionDataset` 클래스와 `custom_collate_fn` 함수를 사용하여 훈련, 검증 및 테스트 데이터 로더를 만듭니다.


<img src="images/llm_from_scratch/ch07_compressed/14.webp" width=700px>


- 이전 `custom_collate_fn` 함수의 또 다른 추가 세부 사항은 메인 학습 루프에서 데이터를 처리하지 않고 타깃 장치(예: GPU)로 직접 이동한다는 것입니다. `custom_collate_fn`을 데이터 로더의 일부로 사용할 때 백그라운드 프로세스로 수행할 수 있으므로 효율성이 향상됩니다.
- 파이썬의 표준 라이브러리 `functools`의 `partial` 함수를 사용하여 원본 함수에서 `device` 매개변수가 미리 채워진 새 함수를 만듭니다.


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

In [ ]:
from functools import partial

customized_collate_fn = partial( #partial은 함수의 기본값을 설정, 원본 함수는 건드리지 않고, 복제본을 만들어서 특정값을 고정
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

- 다음으로, 이전 장과 유사하게 데이터 로더의 인스턴스를 만들지만 이제는 배치 처리를 위해 자체 콜레이트 함수를 전달합니다.


In [ ]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [ ]:
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

- 만들어진 입력 및 타깃 배치의 차원이 어떻게 되는지 살펴보겠습니다.


In [ ]:
print("훈련 데이터 로더:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

- 위 출력 결과에서 볼 수 있듯이, 모든 배치의 크기는 8이지만 예상대로 길이는 다릅니다.
- `inputs` 배치의 첫 번째 훈련 샘플을 출력하여 입력에 토큰 ID 50256에 해당하는 `<|endoftext|>` 패딩 토큰이 포함되어 있는지 다시 한번 확인해 보겠습니다.


In [ ]:
print(inputs[0])

- 마찬가지로, 타깃에 플레이스홀더인 -100 토큰이 포함되어 있는지 다시 확인합니다.


In [ ]:
print(targets[0])

### 📌 노트: LLM 학습 시 패딩(Padding) 처리 방법

#### 1. 입력 데이터 (Input IDs)
* **처리:** 패딩 토큰(예: `50256`)을 **그대로 둡니다.** (절대 `-100`으로 바꾸지 않음)
* **이유:** `Embedding Layer`는 사전(Vocab)에 정의된 인덱스(0 ~ 50256)만 처리할 수 있습니다. `-100`이 입력되면 **IndexError**가 발생합니다.
* **해결:** 모델이 패딩을 읽지 못하게 하려면 **Attention Mask**를 `0`으로 설정하여 가려줍니다.

#### 2. 타겟 데이터 (Target / Label)
* **처리:** 패딩 토큰을 **`-100`으로 치환합니다.**
* **이유:** PyTorch의 `CrossEntropyLoss` 함수는 기본적으로 `-100`인 위치를 **손실(Loss) 계산에서 제외**하기 때문입니다. 이를 통해 불필요한 학습을 방지합니다.

#### 3. 한눈에 보는 비교표

| 구분 | 데이터 값 | 처리 도구 | 목적 |
| :--- | :--- | :--- | :--- |
| **Input** | `50256` (유지) | `Attention Mask` | 모델이 패딩을 **읽지 않게(Attend 하지 않게)** 함 |
| **Target** | `-100` (치환) | `Loss Function` | 모델이 패딩을 **학습(채점)하지 않게** 함 |

## 7.5 사전 훈련된 LLM 로드하기


- 이 절에서는 5장 5.5절과 6장 6.4절에서 사용했던 것과 동일한 코드를 사용하여 사전 훈련된 GPT 모델을 로드합니다.


<img src="images/llm_from_scratch/ch07_compressed/15.webp" width=700px>


- 하지만 1억 2,400만 개의 파라미터를 가진 가장 작은 모델을 로드하는 대신, 3억 5,500만 개의 파라미터를 가진 중간 버전을 로드합니다. 1억 2,400만 개의 파라미터 모델은 지시 미세 튜닝을 통해 합리적인 품질을 얻기에는 너무 작기 때문입니다.


In [ ]:
from previous_chapters import  load_gpt2_model

CHOOSE_MODEL = "gpt2-medium (355M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # 어휘사전 크기
    "context_length": 1024,  # 문맥 길이
    "drop_rate": 0.0,        # 드롭아웃 비율
    "qkv_bias": True         # 쿼리-키-값 편향
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12}, #트랜스포머 블록(Transformer Block)을 몇 층으로 쌓아 올렸는지
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_name = "gpt2-medium-355M.pth"
model = load_gpt2_model(model_name, BASE_CONFIG)
model.eval()

- 다음 절에서 모델 미세 튜닝을 시작하기 전에 유효성 검사 작업의 하나로 모델의 성능을 살펴보겠습니다.


In [ ]:
torch.manual_seed(123)

input_text = format_input(val_data[0])
print(input_text)

In [ ]:
from previous_chapters import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)


token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)

- 이전 장에서 사용했던 `generate` 함수는 입력과 출력 텍스트를 결합하여 반환합니다. 이는 이전 절에서 읽기 쉬운 텍스트를 생성하는 데 편리했습니다.
- 응답을 분리하기 위해 `generated_text`의 시작 부분에서 지시의 길이를 뺄 수 있습니다.


In [ ]:
response_text = (
    generated_text[len(input_text):]
    .replace("### Response:", "")
    .strip()
)
print(response_text)

- 모델이 아직 지시를 따르지 못하는 것을 볼 수 있습니다. "Response" 섹션을 만들지만 원래 입력 문장과 지시를 그대로 반복합니다.


## 7.6 지시 데이터에서 LLM 미세 튜닝하기


- 이 절에서는 모델을 미세 튜닝합니다.

<img src="images/llm_from_scratch/ch07_compressed/16.webp" width=800px>

- 이전 장에서 사용했던 손실 계산 및 학습 함수를 모두 재사용할 수 있습니다.


In [ ]:
from previous_chapters import (
    calc_loss_loader,
    train_model_simple
)

- 훈련을 시작하기 전에 초기 훈련 및 검증 세트 손실을 계산해 보겠습니다(이전 장에서와 같이 목표는 손실을 최소화하는 것입니다).


In [ ]:
model.to(device)

torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("훈련 손실:", train_loss)
print("검증 손실:", val_loss)

- 이전 장보다 더 큰 모델(1억 2,400만 개가 아닌 3억 5,500만 개의 매개변수)을 사용하기 때문에 훈련에 드는 비용이 약간 더 높다는 점에 유의하세요.
- 다양한 기기의 실행 시간은 아래 표를 참고하세요(호환되는 GPU 기기에서 이 노트북을 실행하는 데 코드를 변경할 필요가 없습니다).


<div style="text-align: left;">
    
| 모델              | 기기                | 2 에포크 실행 시간 |
|--------------------|-----------------------|----------------------|
| gpt2-medium (355M) | CPU (M3 MacBook Air)  | 15.78분        |
| gpt2-medium (355M) | GPU (M3 MacBook Air)  | 10.77분        |
| gpt2-medium (355M) | GPU (L4)              | 1.83분         |
| gpt2-medium (355M) | GPU (A100)            | 0.86분         |
| gpt2-small (124M)  | CPU (M3 MacBook Air)  | 5.74분         |
| gpt2-small (124M)  | GPU (M3 MacBook Air)  | 3.73분         |
| gpt2-small (124M)  | GPU (L4)              | 0.69분         |
| gpt2-small (124M)  | GPU (A100)            | 0.39분         |

</div>

- 이 노트북은 `"gpt2-medium (355M)"` 모델을 사용하여 실행했습니다.


In [ ]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"훈련 소요 시간: {execution_time_minutes:.2f}분")

- 위 출력 결과를 바탕으로, 훈련 손실 및 검증 손실 값이 감소하고 있으므로 모델이 잘 훈련되고 있음을 알 수 있습니다.
- 또한 각 에포크 이후 출력된 응답 텍스트를 기반으로, 모델이 입력 문장 `'The chef cooks the meal every day.'` 를 수동태 `'The meal is cooked every day by the chef.'` 로 올바르게 변환하는 지시를 따르고 있음을 확인할 수 있습니다. (응답에 대한 적절한 포맷 및 평가는 이후 절에서 다루겠습니다)
- 마지막으로 훈련 및 검증 손실 곡선을 살펴보겠습니다.


In [ ]:
from previous_chapters import plot_losses

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

- 처음 에포크가 시작될 때 손실이 급격히 감소하는 것을 볼 수 있는데, 이는 모델이 빠르게 학습하기 시작했음을 의미합니다.
- 1 훈련 에포크가 지나면 약간의 과적합이 발생하는 것을 알 수 있습니다.


## 7.7 응답 추출하여 저장하기


<img src="images/llm_from_scratch/ch07_compressed/18.webp" width=700px>


- 이 절에서는 다음 절에서 점수를 매기기 위해 테스트 세트 응답을 저장합니다.
- 또한 나중에 사용할 수 있도록 모델의 복사본을 저장합니다.
- 하지만 먼저, 미세 튜닝된 모델에서 생성된 응답을 간략하게 살펴보겠습니다.


In [ ]:
torch.manual_seed(123)


for entry in test_data[:3]:

    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\n올바른 응답:\n>> {entry['output']}")
    print(f"\n모델 응답:\n>> {response_text.strip()}")
    print("-------------------------------------")

- 테스트 세트 지시와 정답 및 모델의 응답을 기반으로 볼 때 모델은 비교적 잘 수행됩니다.
- 첫 번째 및 마지막 지시에 대한 답변은 명확하게 정확합니다.
- 두 번째 답변은 거의 정확합니다. 모델은 "cumulonimbus" 대신 "cumulus cloud"으로 답합니다(그러나 적운은 천둥번개를 일으킬 수 있는 적란운으로 발전할 수 있습니다).
- 가장 중요한 것은 이전 장에서 정확히 맞은 스팸/스팸아닌 클래스 레이블의 비율을 계산하여 분류 정확도를 얻었던 것처럼 모델 평가가 간단하지 않다는 것입니다.
- 실제로 챗봇과 같은 지시 미세 튜닝 LLM은 여러 접근 방식을 통해 평가됩니다.
  - 모델의 지식을 테스트하는 MMLU("Measuring Massive Multitask Language Understanding", [https://arxiv.org/abs/2009.03300](https://arxiv.org/abs/2009.03300))와 같은 짧은 답변 및 선다형 벤치마크
  - LMSYS 챗봇 아레나([https://arena.lmsys.org](https://arena.lmsys.org))와 같은 여러 LLM에 대한 인간의 선호도 비교
  - AlpacaEval([https://tatsu-lab.github.io/alpaca_eval/](https://tatsu-lab.github.io/alpaca_eval/))과 같이 GPT-4와 같은 다른 LLM을 사용하여 응답을 평가하는 자동화된 대화 벤치마크

- 다음 절에서는 AlpacaEval과 유사한 접근 방식을 사용해 다른 LLM으로 모델의 응답을 평가합니다. 그러나 공개적으로 사용 가능한 벤치마크 데이터셋을 사용하는 대신 자체 테스트 세트를 사용합니다.
- 이를 위해 모델 응답을 `test_data` 딕셔너리에 추가하고 기록 보관을 위해 `"instruction-data-with-response.json"` 파일로 저장하여 필요한 경우 별도의 파이썬 세션에서 로드하고 분석할 수 있도록 합니다.


In [ ]:
from tqdm import tqdm

for i, entry in tqdm(enumerate(test_data), total=len(test_data)):

    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = generated_text[len(input_text):].replace("### Response:", "").strip()

    test_data[i]["model_response"] = response_text


with open("outputs/instruction-data-with-response.json", "w") as file:
    json.dump(test_data, file, indent=4)  # 미려한 출력을 위해 "indent" 사용

- `test_data` 딕셔너리에 응답이 제대로 추가되었는지 확인하기 위해 항목 중 하나를 다시 확인해 보겠습니다.


In [ ]:
print(test_data[0])

- 마지막으로, 나중에 재사용할 수 있도록 모델을 저장합니다.


In [ ]:
import re


file_name = f"outputs/{re.sub(r'[ ()]', '', CHOOSE_MODEL) }-sft.pth"
torch.save(model.state_dict(), file_name)
print(f"모델이 {file_name}에 저장되었습니다.")

# 모델 로드 방법:
# model.load_state_dict(torch.load("outputs/gpt2-medium355M-sft.pth"))

## 7.8 미세 튜닝된 LLM 평가하기


**이 절은 로컬에서 Ollama를 설치한 후 주피터 노트북에서 이어서 실행할 수 있습니다**

<img src="images/llm_from_scratch/ch07_compressed/19.webp" width=700px>


- gpt로 모델을 평가하는 방법에 대해 관심이 있으시면 [llm-instruction-eval-openai.ipynb](llm-instruction-eval-openai.ipynb)을 참고하세요.

## 7.9 결론


### 7.9.1 다음 단계는?

- 이 장은 이 책의 마지막 장입니다.
- LLM 개발 사이클의 주요 단계인 LLM 구조 구현, LLM 사전 훈련 및 미세 튜닝을 다루었습니다.

<img src="images/llm_from_scratch/ch07_compressed/21.webp" width=800px>

- 이 장에서 설명한 지시 미세 튜닝 후에 수행되는 선택적 단계는 선호도 미세 튜닝입니다.
- 선호도 미세 튜닝 프로세스는 특정 사용자 선호도에 맞게 모델을 커스터마이징하는 데 특히 유용할 수 있습니다. 관심 있는 경우 [../04_preference-tuning-with-dpo](../04_preference-tuning-with-dpo) 폴더를 참조하세요.

- 이 깃허브 저장소에는 다양한 추가 보너스 자료가 포함되어 있습니다. 자세한 내용은 이 저장소의 README 페이지에 있는 [보너스 자료](https://github.com/rickiepark/llm-from-scratch?tab=readme-ov-file#bonus-material) 섹션을 참조하세요.

### 7.9.2 빠르게 변화하는 분야의 최신 정보 얻기

- 이 섹션에는 코드가 없습니다.

### 7.9.3 맺음말

- 처음부터 LLM을 구현하고 사전 훈련 및 미세 튜닝 함수를 코딩하는 이 여정을 즐겼기를 바랍니다.
- 제 생각에는 처음부터 LLM을 구현하는 것이 LLM의 작동 방식을 이해하는 가장 좋은 방법입니다. 이 접근 방식을 통해 더 잘 이해했기를 바랍니다.
- 이 책은 교육 목적을 위해 작성되었지만 실제 애플리케이션을 위해 더 강력한 다른 LLM을 사용하는 데 관심이 있을 수 있습니다.
  - 이를 위해 axolotl([https://github.com/OpenAccess-AI-Collective/axolotl](https://github.com/OpenAccess-AI-Collective/axolotl)) 또는 제가 개발을 돕고 있는 LitGPT([https://github.com/Lightning-AI/litgpt](https://github.com/Lightning-AI/litgpt))와 같은 인기 있는 도구를 고려할 수 있습니다.


## 요약

- 독립적인 지시 미세 튜닝 스크립트는 [./gpt_instruction_finetuning.py](./gpt_instruction_finetuning.py)를 참조하세요.
- [./ollama_evaluate.py](./ollama_evaluate.py)는 7.8절을 기반으로 "output" 및 "response" 키가 포함된 JSON 파일을 Ollama 및 Llama 3를 통해 평가하는 독립형 스크립트입니다.
- [./load-finetuned-model.ipynb](./load-finetuned-model.ipynb) 노트북은 새로운 세션에서 미세 튜닝된 모델을 로드하는 방법을 보여줍니다.
- [./exercise-solutions.ipynb](./exercise-solutions.ipynb)에서 연습 문제 솔루션을 확인할 수 있습니다.


## 이어서 볼 만한 자료

- 책을 완료하신 것을 축하드립니다. 추가 자료를 찾고 계신 경우를 대비하여, 이 깃허브 저장소에 흥미로울 만한 몇 가지 보너스 섹션을 추가했습니다.
- 보너스 자료의 전체 목록은 메인 README의 [보너스 자료](https://github.com/rickiepark/llm-from-scratch?tab=readme-ov-file#bonus-material) 섹션에서 볼 수 있습니다.
- 제가 좋아하는 몇 가지를 강조하자면 다음과 같습니다.
  1. [(밑바닥부터) LLM 정렬을 위한 DPO](../04_preference-tuning-with-dpo/dpo-from-scratch.ipynb)는 이 장의 모델을 인간의 선호도에 더 가깝게 정렬하기 위한 인기 있는 선호도 튜닝 메커니즘을 구현합니다.
  2. [처음부터 Llama 3.2(독립형 노트북)](../../ch05/07_gpt_to_llama/standalone-llama32.ipynb)는 공식 사전 훈련된 가중치 로드를 포함하여 메타 AI의 인기 있는 Llama 3.2를 처음부터 구현한 것입니다. 추가 실험을 하고 싶다면 각 장의 `GPTModel` 모델을 `Llama3Model` 클래스로 대체할 수 있습니다.
  3. [GPT를 Llama로 변환](../../ch05/07_gpt_to_llama)에는 GPT-2와 다양한 Llama 모델 간의 차이점을 설명하는 단계별 가이드가 포함된 코드가 있습니다.
  4. [임베딩 층과 선형 층의 차이점 이해](../../ch02/03_bonus_embedding-vs-matmul/embeddings-and-linear-layers.ipynb)는 LLM의 입력 단계에서 사용하는 파이토치의 `Embedding` 층이 수학적으로 원-핫 인코딩된 데이터에 적용된 선형 층과 동일하다는 것을 보여주는 개념적 설명입니다.
- 더 즐거운 시간이 되시기 바랍니다!
